In [2]:
import pandas as pd

In [3]:
df=pd.read_csv('IMDB Dataset.csv')

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

df.shape

# Pre-processing

## Converting to lowercase

In [7]:
df['review']=df['review'].str.lower()

## Removing the URL'S

In [7]:
import re
def removeURL(text):
    text=re.sub(r"http:/S+","",text)
    return text


In [8]:
df['review']=df['review'].apply(removeURL)

## Removing the punctuations

In [9]:
def removePunctuation(text):
    text=re.sub(r"[^A-Za-z0-9]\s","",text)
    return text
    

In [10]:
df['review']=df['review'].apply(removePunctuation)

## Removing the HTML

In [11]:
def removeHTML(text):
    text=re.sub(r"<.*?>","",text)
    return text
    

In [12]:
df['review']=df['review'].apply(removeHTML)

## Removing the stopwords

In [13]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to C:\Users\SHREYASH
[nltk_data]     RAUT\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\SHREYASH
[nltk_data]     RAUT\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\SHREYASH
[nltk_data]     RAUT\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
def removestopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text
            
df['review']=df['review'].apply(removestopwords)

In [17]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode 'll h...,positive
1,wderful ltle producti filming technique uns...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly 's fmly lttle boy (jkethks 's zombe...,negative
4,"petter mttei's ""love time mey vully stunng...",positive


## Stemming

In [18]:
from nltk.stem import PorterStemmer


In [19]:
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words) #converting to string

df['review']=df['review'].apply(stemming)

In [20]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod 'll hookedi rght ...,positive
1,wder ltle producti film techniqu unssum old-ti...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli 's fmli lttle boy ( jkethk 's zomb clos...,negative
4,petter mttei 's `` love time mey vulli stunng ...,positive


## Encoding

In [16]:
from sklearn.preprocessing import LabelEncoder

In [17]:
le= LabelEncoder()
df['sentiment']=le.fit_transform(df['sentiment'])
y=df['sentiment']

## Vectorization

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
tf = TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df['review'])

In [25]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3638588 stored elements and shape (49582, 5000)>

## Dataset and DataLoader

In [20]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)



In [21]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [22]:
X_train=X_train.toarray() #converting it to numpy array as it is in sparse matrix form, because tensors are created using numpy arrays
X_test=X_test.toarray()

In [23]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [24]:
train_loader=DataLoader(train_set,batch_size=64,shuffle=True)
test_loader=DataLoader(test_set,batch_size=64,shuffle=True)

## RNN

In [31]:
import torch.nn as nn
import torch.optim as optim

In [32]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layers=num_layers

        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        self.fc=nn.Linear(hidden_size,1) #Fully Connected layer)

    def forward(self,x):
            #optional 
            h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
            out,_=self.rnn(x,h0) #out returns hidden state  of all timesteps  (batch,seq_len,hidden_size)
            out=self.fc(out[:,-1,:]) #We need only last value of seq_len that's why -1 is written
            return out
            
            
    

In [33]:
input_size=X_train.shape[1]
model = RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [34]:
X_train.shape


(39665, 5000)

## Training and Evaluation

In [36]:
epochs=10
for epoch in range(epochs):
    model.train()
    for Xb,yb in train_loader:
        optimizer.zero_grad()
        Xb=Xb.unsqueeze(1) # converting the Xb into 3 dimension from 2 using unsqueeze as model expects it in 3 dimension
        output=model(Xb) # (batch size,1)
        output = torch.sigmoid(output.squeeze()) #To calculate the probabilities single dimension is needed so convert 2 dimension into 1 using squeeze. (batch size,)=> probability
        loss = criterion(output,yb)
        loss.backward()
        optimizer.step()
    print(f"epoch={epoch+1} and loss ={loss}")
        

epoch=1 and loss =0.5565247535705566
epoch=2 and loss =0.299157053232193
epoch=3 and loss =0.5166836977005005
epoch=4 and loss =0.14574863016605377
epoch=5 and loss =0.48520737886428833
epoch=6 and loss =0.17025646567344666
epoch=7 and loss =0.2409522980451584
epoch=8 and loss =0.26472702622413635
epoch=9 and loss =0.2773893177509308
epoch=10 and loss =0.37180787324905396


In [37]:
model.eval()

with torch.no_grad():
    correct_val=0
    total_val=0
    for Xb,yb in test_loader:
        Xb=Xb.unsqueeze(1)
        output = model(Xb)
        predicted =(torch.sigmoid(output.squeeze()) > 0.5).float()
        correct_val += (predicted == yb).sum().item()
        total_val += yb.size(0)
    print(f"Accuracy = {correct_val/total_val}")

Accuracy = 0.845517797721085


In [26]:
for Xb,yb in train_loader:
    print(Xb.shape)
    break

torch.Size([64, 5000])
